In [0]:
base_path = "/Volumes/dbdemos/data/healthcare_lab_data"
catalog = "dbdemos"
schema = "silver"

tables = ["sample", "test", "lot", "result", "item"]

for table in tables:
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(f"{base_path}{table}.csv")
    )

    df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table}")

    print(f"Silver table created: {catalog}.{schema}.{table}")

In [0]:
df = spark.sql("""
WITH base AS (
    SELECT
        SUBSTRING(A.label_id, 1, 11)       AS lot,
        SUBSTRING(A.label_id, 1, 10)       AS join_id,
        'tall'                             AS site,
        'mall'                             AS category,
        A.product,
        F.ITEM_DESCRIPTION                 AS lot_description,
        E.entry                            AS result_value,
        E.units,
        B.date_completed                   AS result_captured_date
    FROM dbdemos.silver.sample A
    LEFT JOIN dbdemos.silver.result E
        ON A.sample_number = E.sample_number
    LEFT JOIN dbdemos.silver.test B
        ON E.test_number = B.test_number
    LEFT JOIN dbdemos.silver.lot C
        ON C.lot_name = SUBSTRING(A.label_id, 1, 11)
    LEFT JOIN dbdemos.silver.item F
        ON F.lot_number = C.lot_name
    WHERE E.name IS NOT NULL
      AND E.status = 'A'
      AND B.status = 'A'
      AND E.entry IS NOT NULL
),

latest AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY lot
            ORDER BY result_captured_date DESC
        ) AS rnk
    FROM base
)

SELECT
    product,
    lot_description,
    site,
    join_id,
    lot,
    category,
    result_value AS entry,
    units,
    result_captured_date,
    CURRENT_TIMESTAMP() AS EDL_INRT_DTTM,
    CURRENT_TIMESTAMP() AS EDL_LAST_MODF_DTTM
FROM latest
WHERE rnk = 1
  AND result_value IS NOT NULL

""")

df.write.mode("overwrite").saveAsTable("dbdemos.gold.fact_results")